# Notebook 2 — MASTER Baseline Feature Selection v3
## Healthcare-Security Feature-Selection Benchmark — 4-Dataset Robust Master

**Purpose:** one reproducible baseline protocol for HIPAA, WUSTL, CICIOMT, and WUSTL_HDRL.

### v3 upgrades
- Single canonical **30-method** panel for all four datasets.
- Batch execution of all four datasets in one notebook.
- Per-method exception isolation, traceback logging, and checkpoint/resume.
- Data-quality, leakage/proxy, class-balance, feature-space, and reproducibility audits.
- Resource-safe parallelism to reduce RAM pressure and nested-joblib warnings.
- Safe handling of empty/constant/non-finite features and degenerate statistical cases.
- Dataset-level and cross-dataset manifests for NB3/NB4.
- Scientific integrity gate: failures are recorded, never silently converted into valid results.

**Important:** a method failure is never fabricated into a result. The notebook continues, records the failure, and the final audit reports the run as incomplete.


## Section 0 — Environment and dependency audit

In [1]:
import os, sys, json, time, traceback, warnings, platform, importlib.util
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import MinMaxScaler, KBinsDiscretizer, LabelEncoder
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_selection import (chi2, f_classif, mutual_info_classif,
    VarianceThreshold, RFE, RFECV, SelectFromModel)
from sklearn.inspection import permutation_importance
from sklearn.base import clone
from scipy.stats import spearmanr

# Do not globally suppress warnings: warnings are part of the audit trail.
warnings.filterwarnings('default')

AUTO_INSTALL_MISSING = True
REQUIRED_PACKAGES = {
    'mlxtend': 'mlxtend', 'skrebate': 'scikit-rebate', 'boruta': 'Boruta',
    'BorutaShap': 'BorutaShap', 'lightgbm': 'lightgbm', 'mrmr': 'mrmr-selection',
    'xgboost': 'xgboost', 'shap': 'shap', 'zoofs': 'zoofs'
}

def dependency_audit():
    missing=[]
    for module,pkg in REQUIRED_PACKAGES.items():
        if importlib.util.find_spec(module) is None:
            missing.append((module,pkg))
    if missing and AUTO_INSTALL_MISSING:
        import subprocess
        print('Installing missing optional dependencies:', [p for _,p in missing])
        try:
            subprocess.check_call([sys.executable,'-m','pip','install','-q']+[p for _,p in missing])
        except Exception as e:
            print('Dependency installation failed:', type(e).__name__, e)
    status={m: importlib.util.find_spec(m) is not None for m in REQUIRED_PACKAGES}
    print('Dependency status:', status)
    return status

DEPENDENCY_STATUS = dependency_audit()
print('Python:', sys.version.split()[0], '| Platform:', platform.platform())


Dependency status: {'mlxtend': True, 'skrebate': True, 'boruta': True, 'BorutaShap': True, 'lightgbm': True, 'mrmr': True, 'xgboost': True, 'shap': True, 'zoofs': True}
Python: 3.13.9 | Platform: Windows-11-10.0.26200-SP0


## Section 1 — Master scientific configuration

In [2]:
BENCH_ROOT = r'C:\Users\tanvi\Downloads\HealthSec_FS'
DATASETS = ['HIPAA','WUSTL','CICIOMT','WUSTL_HDRL']
DATASET_TARGETS = {d:'is_attack' for d in DATASETS}

# Scientific controls: identical across datasets.
RANDOM_STATE = 42

PROTOCOL_VERSION = "NB2_MASTER_SCIENTIFIC_v3"
NOTEBOOK_VERSION = "v3"
BENCHMARK_NAME = "HealthSec_FS"

CV_FOLDS = 5
PRIMARY_METRIC = 'mcc'
SCORING = 'matthews_corrcoef'

K_FLOOR = 5
CV_SELECTION_STEP = 1
PVALUE_ALPHA = 0.05
CUMULATIVE_IMPORTANCE_FRACTION = 0.95
PERMUTATION_REPEATS = 10
NEIGHBOR_MAX_ROWS = 4000
META_POP = 25
META_ITERS = 40
N_JOBS = 1   # resource-safe; avoids nested joblib oversubscription/warnings
RESUME = True
SAVE_CHECKPOINT_EVERY_METHOD = True
STRICT_FINAL_AUDIT = False  # False = continue/report; True = raise on incomplete dataset

RUN_METHODS = {
 'chi2':True,'anova':True,'mutual_info':True,'info_gain':True,'gain_ratio':True,'fisher':True,'reliefF':True,'variance':True,'spearman':True,'fcbf':True,'laplacian':True,
 'rfe':True,'rfecv':True,'sfs':True,'sbs':True,'ga':True,'bpso':True,'bgwo':True,
 'lasso':True,'elasticnet':True,'rf_importance':True,'extratrees_importance':True,'dtree_importance':True,'xgb_importance':True,'lgbm_importance':True,'permutation':True,
 'mrmr':True,'boruta':True,'borutashap':True,'shap_rank':True}
EXPECTED_METHOD_COUNT=30
FAMILY={**{m:'filter' for m in ['chi2','anova','mutual_info','info_gain','gain_ratio','fisher','reliefF','variance','spearman','fcbf','laplacian']},
 **{m:'wrapper' for m in ['rfe','rfecv','sfs','sbs','ga','bpso','bgwo']},
 **{m:'embedded' for m in ['lasso','elasticnet','rf_importance','extratrees_importance','dtree_importance','xgb_importance','lgbm_importance','permutation']},
 **{m:'hybrid' for m in ['mrmr','boruta','borutashap','shap_rank']}}
assert len(RUN_METHODS)==EXPECTED_METHOD_COUNT==len(FAMILY)==30
assert all(RUN_METHODS.values())
print('Canonical 30-method panel validated.')


Canonical 30-method panel validated.


In [3]:
# ============================================================
# MASTER PROTOCOL IDENTIFICATION
# ============================================================

PROTOCOL_VERSION = "NB2_MASTER_SCIENTIFIC_v3"
NOTEBOOK_VERSION = "v3"
BENCHMARK_NAME = "HealthSec_FS"

print("Protocol version:", PROTOCOL_VERSION)
print("Notebook version:", NOTEBOOK_VERSION)
print("Benchmark:", BENCHMARK_NAME)

Protocol version: NB2_MASTER_SCIENTIFIC_v3
Notebook version: v3
Benchmark: HealthSec_FS


## Section 2 — Robust dataset loader and audit

In [4]:
CURRENT_DATASET=None
PROJECT_ROOT=None
PATHS={}
TRAIN_PATH=None
TEST_PATH=None
TARGET_COLUMN=None
X=None
y=None
ALL_FEATURES=[]
CV=None
RESULTS=[]
FAILURES=[]
WARNING_LOG=[]

def _make_paths(dataset):
    root=os.path.join(BENCH_ROOT,dataset)
    paths={n:os.path.join(root,n) for n in ['data','processed','figures','models','results','logs']}
    for p in paths.values(): os.makedirs(p,exist_ok=True)
    return root,paths

def _encode_target(s):
    s=s.copy()
    if s.isna().any():
        raise ValueError(f'Target contains {int(s.isna().sum())} missing values.')
    if s.dtype==bool: return s.astype(int)
    if pd.api.types.is_numeric_dtype(s):
        vals=sorted(pd.unique(s))
        if set(vals)=={0,1}: return s.astype(int)
        if len(vals)==2: return pd.Series(LabelEncoder().fit_transform(s),index=s.index,name=s.name)
        raise ValueError(f'Expected binary target; found {len(vals)} numeric classes: {vals[:10]}')
    return pd.Series(LabelEncoder().fit_transform(s.astype(str)),index=s.index,name=s.name)

def load_dataset(dataset):
    global CURRENT_DATASET,PROJECT_ROOT,PATHS,TRAIN_PATH,TEST_PATH,TARGET_COLUMN,X,y,ALL_FEATURES,CV,RESULTS,FAILURES,WARNING_LOG
    CURRENT_DATASET=dataset; PROJECT_ROOT,PATHS=_make_paths(dataset); TARGET_COLUMN=DATASET_TARGETS[dataset]
    TRAIN_PATH=os.path.join(PATHS['processed'],'train.parquet'); TEST_PATH=os.path.join(PATHS['processed'],'test.parquet')
    if not os.path.exists(TRAIN_PATH): raise FileNotFoundError(f'Missing NB1 training file: {TRAIN_PATH}')
    df=pd.read_parquet(TRAIN_PATH)
    if TARGET_COLUMN not in df.columns: raise KeyError(f"Target '{TARGET_COLUMN}' missing. Columns={df.columns.tolist()}")
    y=_encode_target(df[TARGET_COLUMN])
    X=df.drop(columns=[TARGET_COLUMN]).copy()
    # One-hot encoding is dataset-local; NB1 should already provide processed data.
    obj_cols=X.select_dtypes(include=['object','category']).columns.tolist()
    if obj_cols: X=pd.get_dummies(X,columns=obj_cols,dtype=float)
    X=X.apply(pd.to_numeric,errors='coerce')
    X=X.replace([np.inf,-np.inf],np.nan)
    # Remove all-missing and constant columns before FS.
    all_missing=[c for c in X.columns if X[c].isna().all()]
    if all_missing: X=X.drop(columns=all_missing)
    X=X.fillna(X.median(numeric_only=True))
    X=X.fillna(0.0)
    constant=[c for c in X.columns if X[c].nunique(dropna=False)<=1]
    if constant: X=X.drop(columns=constant)
    X=X.astype(float)
    ALL_FEATURES=list(X.columns)
    if len(ALL_FEATURES)<K_FLOOR: raise ValueError(f'Only {len(ALL_FEATURES)} usable features; K_FLOOR={K_FLOOR}.')
    counts=y.value_counts().sort_index()
    if len(counts)!=2: raise ValueError(f'Binary benchmark requires 2 classes; found {counts.to_dict()}')
    if counts.min()<CV_FOLDS: raise ValueError(f'Minority class has {counts.min()} rows; cannot run {CV_FOLDS}-fold stratified CV.')
    CV=StratifiedKFold(n_splits=CV_FOLDS,shuffle=True,random_state=RANDOM_STATE)
    RESULTS=[]; FAILURES=[]; WARNING_LOG=[]
    audit=pd.DataFrame({
      'feature':ALL_FEATURES,
      'missing_after_cleaning':[int(X[c].isna().sum()) for c in ALL_FEATURES],
      'nunique':[int(X[c].nunique()) for c in ALL_FEATURES],
      'dtype':[str(X[c].dtype) for c in ALL_FEATURES]})
    audit.to_csv(os.path.join(PATHS['results'],'nb2_data_quality_audit.csv'),index=False)
    print(f'\n[{dataset}] rows={len(X):,} features={len(ALL_FEATURES)} classes={counts.to_dict()}')
    print('Train:',TRAIN_PATH,'| Test exists:',os.path.exists(TEST_PATH))
    return df

def leakage_audit():
    rows=[]
    yy=y.reset_index(drop=True)
    for c in ALL_FEATURES:
        s=X[c].reset_index(drop=True)
        exact=bool(np.array_equal(s.values,yy.values)) if len(s)==len(yy) else False
        perfect=False
        if s.nunique()==2:
            tab=pd.crosstab(s,yy)
            perfect=(tab.shape==(2,2) and all((tab.max(axis=0)==tab.sum(axis=0))))
        rows.append({'feature':c,'nunique':int(s.nunique()),'exact_target_duplicate':exact,'perfect_binary_association':bool(perfect)})
    out=pd.DataFrame(rows); out.to_csv(os.path.join(PATHS['results'],'nb2_leakage_audit.csv'),index=False)
    flags=out[out[['exact_target_duplicate','perfect_binary_association']].any(axis=1)]
    print('Leakage/proxy flags:',len(flags))
    if len(flags): print(flags.to_string(index=False))
    return out


## Section 3 — Shared utilities

In [5]:
def probe(): return LogisticRegression(max_iter=1000,random_state=RANDOM_STATE)

def _checkpoint_path(): return os.path.join(PATHS['results'],'nb2_checkpoint.json')

def _save_checkpoint():
    payload={'dataset':CURRENT_DATASET,'timestamp':time.strftime('%Y-%m-%d %H:%M:%S'),'results':RESULTS,'failures':FAILURES,'warnings':WARNING_LOG}
    with open(_checkpoint_path(),'w',encoding='utf-8') as f: json.dump(payload,f,indent=2,default=str)

def _load_checkpoint():
    p = _checkpoint_path()
    if not RESUME or not os.path.exists(p): return
    try:
        with open(p, encoding='utf-8') as f:
            payload = json.load(f)
        if payload.get('dataset') == CURRENT_DATASET:
            RESULTS.extend(payload.get('results', []))
            FAILURES.extend(payload.get('failures', []))
            WARNING_LOG.extend(payload.get('warnings', []))
            print(f'Resumed checkpoint: {len(RESULTS)} completed, {len(FAILURES)} failures.')
    except Exception as e:
        print('Checkpoint ignored:', type(e).__name__, e)

def register(name,family,selected,justification,fit_seconds=None,warning_count=0):
    selected=list(dict.fromkeys(selected))
    invalid=[f for f in selected if f not in ALL_FEATURES]
    if invalid: raise ValueError(f'{name}: invalid selected features: {invalid[:5]}')
    if not selected: raise ValueError(f'{name}: empty feature subset')
    RESULTS.append({'method':name,'family':family,'n_selected':len(selected),'selected_features':selected,'justification':justification,'fit_time_seconds':fit_seconds,'warning_count':warning_count,'status':'success'})
    print(f'[{family:<8}] {name:<22} -> {len(selected):>2} features | {justification} [{fit_seconds:.1f}s]')

def safe_run(name,family,fn):
    global RESULTS,FAILURES,WARNING_LOG
    if not RUN_METHODS.get(name,False): return
    if any(r.get('method')==name and r.get('status')=='success' for r in RESULTS):
        print(f'[{family:<8}] {name:<22} -> resumed; already complete'); return
    t0=time.time(); caught=[]
    with warnings.catch_warnings(record=True) as ws:
        warnings.simplefilter('always')
        try:
            selected,justification=fn()
            for w in ws: caught.append({'method':name,'category':w.category.__name__,'message':str(w.message)})
            WARNING_LOG.extend(caught)
            register(name,family,selected,justification,time.time()-t0,len(caught))
        except Exception as e:
            for w in ws: caught.append({'method':name,'category':w.category.__name__,'message':str(w.message)})
            WARNING_LOG.extend(caught)
            failure={'method':name,'family':family,'error_type':type(e).__name__,'error':str(e),'traceback':traceback.format_exc(),'fit_time_seconds':time.time()-t0}
            FAILURES.append(failure)
            print(f'[{family:<8}] {name:<22} -> ERROR: {failure["error_type"]}: {failure["error"]}')
            with open(os.path.join(PATHS['logs'],f'nb2_{name}_error.txt'),'w',encoding='utf-8') as f: f.write(failure['traceback'])
    if SAVE_CHECKPOINT_EVERY_METHOD: _save_checkpoint()

def _neighbor_sample():
    if len(X)<=NEIGHBOR_MAX_ROWS: return X,y
    idx,_=train_test_split(np.arange(len(X)),train_size=NEIGHBOR_MAX_ROWS,stratify=y,random_state=RANDOM_STATE)
    return X.iloc[idx],y.iloc[idx]

def select_k_by_cv(ranked,est=None,step=CV_SELECTION_STEP):
    ranked=[f for f in ranked if f in ALL_FEATURES]
    if not ranked: raise ValueError('Ranking is empty')
    est=est or probe(); ks=list(range(1,len(ranked)+1,step))
    if ks[-1]!=len(ranked): ks.append(len(ranked))
    means=[]; stds=[]
    for k in ks:
        s=cross_val_score(clone(est),X[ranked[:k]],y,cv=CV,scoring=SCORING,error_score='raise')
        means.append(float(np.mean(s))); stds.append(float(np.std(s,ddof=1)) if len(s)>1 else 0.0)
    best_i=int(np.nanargmax(means)); se=stds[best_i]/np.sqrt(CV_FOLDS); thresh=means[best_i]-se
    chosen_k=next(ks[i] for i in range(len(ks)) if means[i]>=thresh)
    chosen_k=min(max(chosen_k,K_FLOOR),len(ranked))
    return ranked[:chosen_k],f'k={chosen_k} by {CV_FOLDS}-fold CV {SCORING} 1-SE rule (peak={means[best_i]:.3f}, SE={se:.3f}, floor={K_FLOOR})'

def select_by_cumulative(importances,frac=CUMULATIVE_IMPORTANCE_FRACTION):
    imp=np.nan_to_num(np.asarray(importances,dtype=float),nan=0.0,posinf=0.0,neginf=0.0)
    if imp.sum()<=0: return [ALL_FEATURES[int(np.argmax(imp))]],'fallback to highest-ranked feature because total importance was zero'
    order=np.argsort(imp)[::-1]; ranked=[ALL_FEATURES[i] for i in order]
    cum=np.cumsum(imp[order])/(imp.sum()+1e-12); k=int(np.searchsorted(cum,frac)+1)
    return ranked[:max(1,k)],f'k={k} keeps features covering {int(frac*100)}% of total importance'

def select_by_pvalue(pvals,alpha=PVALUE_ALPHA):
    p=np.nan_to_num(np.asarray(pvals,dtype=float),nan=1.0,posinf=1.0,neginf=0.0)
    keep=[ALL_FEATURES[i] for i in range(len(p)) if p[i]<alpha]
    if not keep: keep=[ALL_FEATURES[int(np.argmin(p))]]; return keep,f'no p<{alpha}; retained minimum-p feature as deterministic fallback'
    return keep,f'features with p < {alpha}'


## Section 4–6 — Canonical 30 feature-selection methods

In [6]:
# Chi-Square: requires NON-NEGATIVE features, so we min-max scale a copy for this test only.
def m_chi2():
    Xs = pd.DataFrame(MinMaxScaler().fit_transform(X), columns=ALL_FEATURES)
    stat, pvals = chi2(Xs, y)
    return select_by_pvalue(pvals)


In [7]:
# ANOVA F-test: relevance of each (continuous) feature to the class via F-statistic p-values.
def m_anova():
    F, pvals = f_classif(X, y)
    return select_by_pvalue(pvals)


In [8]:
# Mutual Information: non-linear dependence between feature and target; rank then CV-pick k.
def m_mi():
    mi = mutual_info_classif(X, y, random_state=RANDOM_STATE)
    ranked = [ALL_FEATURES[i] for i in np.argsort(mi)[::-1]]
    return select_k_by_cv(ranked)


In [9]:
# Information Gain: entropy reduction H(Y)-H(Y|X) on discretized features; rank then CV-pick k.
def _entropy(p):
    p = p[p > 0]; return -np.sum(p*np.log2(p))
def _discretize(col, bins=10):
    if col.nunique() <= bins: return col.astype(int).values
    return KBinsDiscretizer(bins, encode='ordinal', strategy='quantile'
            ).fit_transform(col.values.reshape(-1,1)).ravel().astype(int)
def _info_gain_col(xd, y):
    Hy = _entropy(np.bincount(y)/len(y))
    Hyx = 0.0
    for v in np.unique(xd):
        idx = xd == v; w = idx.mean()
        Hyx += w * _entropy(np.bincount(y[idx], minlength=y.max()+1)/idx.sum())
    return Hy - Hyx
def m_info_gain():
    yv = y.values
    ig = np.array([_info_gain_col(_discretize(X[c]), yv) for c in ALL_FEATURES])
    ranked = [ALL_FEATURES[i] for i in np.argsort(ig)[::-1]]
    return select_k_by_cv(ranked)


In [10]:
# Gain Ratio: Information Gain normalized by the feature's own entropy (intrinsic value).
def m_gain_ratio():
    yv = y.values
    gr = []

    for c in ALL_FEATURES:
        xd = np.asarray(_discretize(X[c]), dtype=int)

        # Gain Ratio requires non-negative discrete category labels
        xd = xd - xd.min()

        ig = _info_gain_col(xd, yv)

        counts = np.bincount(xd)
        iv = _entropy(counts / counts.sum()) + 1e-12

        gr.append(ig / iv)

    gr = np.asarray(gr, dtype=float)

    ranked = [
        ALL_FEATURES[i]
        for i in np.argsort(gr)[::-1]
    ]

    return select_k_by_cv(ranked)




In [11]:
# Fisher Score: between-class separation over within-class scatter; rank then CV-pick k.
def m_fisher():
    yv = y.values; classes = np.unique(yv)
    scores = []
    for c in ALL_FEATURES:
        xc = X[c].values; mu = xc.mean()
        num = den = 0.0
        for cl in classes:
            xi = xc[yv == cl]
            num += len(xi)*(xi.mean()-mu)**2
            den += len(xi)*xi.var()
        scores.append(num/(den+1e-12))
    ranked = [ALL_FEATURES[i] for i in np.argsort(scores)[::-1]]
    return select_k_by_cv(ranked)


In [12]:
# ReliefF: instance-based relevance that captures feature interactions (skrebate).
def m_relieff():
    from skrebate import ReliefF
    Xn, yn = _neighbor_sample()
    r = ReliefF(n_neighbors=max(1, min(10, int(yn.value_counts().min()-1))))
    r.fit(Xn.values, yn.values)
    ranked = [ALL_FEATURES[i] for i in np.argsort(r.feature_importances_)[::-1]]
    return select_k_by_cv(ranked)


In [13]:
# Variance Threshold: drop quasi-constant features (almost no information).
def m_variance():
    vt = VarianceThreshold(threshold=0.01).fit(MinMaxScaler().fit_transform(X))
    keep = [ALL_FEATURES[i] for i in range(len(ALL_FEATURES)) if vt.get_support()[i]]
    return keep, "removed near-constant features (min-max variance < 0.01)"


In [14]:
# Spearman: monotonic (rank) correlation with the target; ordinal-safe relevance ranking.
def m_spearman():
    yv = y.values
    corr = np.array([abs(spearmanr(X[c].values, yv).correlation) for c in ALL_FEATURES])
    corr = np.nan_to_num(corr)
    ranked = [ALL_FEATURES[i] for i in np.argsort(corr)[::-1]]
    return select_k_by_cv(ranked)


In [15]:
# FCBF (Fast Correlation-Based Filter): keep predominant features, remove redundant ones
# using Symmetric Uncertainty (SU). Threshold = mean SU with the target (auto, no manual k).
def _su(a, b):
    a = _discretize(pd.Series(a)) if len(np.unique(a)) > 12 else a.astype(int)
    Ha = _entropy(np.bincount(a-a.min())/len(a))
    Hb = _entropy(np.bincount(b)/len(b))
    # I(a;b) via joint
    ig = Hb
    for v in np.unique(a):
        idx = a == v; w = idx.mean()
        ig -= w*_entropy(np.bincount(b[idx], minlength=b.max()+1)/idx.sum())
    return 2*ig/(Ha+Hb+1e-12)
def m_fcbf():
    yv = y.values
    su_t = {c: _su(X[c].values, yv) for c in ALL_FEATURES}
    thr = np.mean(list(su_t.values()))
    cand = sorted([c for c in ALL_FEATURES if su_t[c] >= thr], key=lambda c:-su_t[c])
    selected = []
    while cand:
        fp = cand.pop(0); selected.append(fp)
        cand = [c for c in cand if _su(X[c].values, _discretize(X[fp]).astype(int)+0) < su_t[c]]
    return selected, f"SU>={thr:.3f} with target, then redundant features removed (FCBF)"


In [16]:
# Laplacian Score: unsupervised; favors features that preserve local neighborhood structure.
def m_laplacian():
    from sklearn.neighbors import kneighbors_graph
    Xn, _ = _neighbor_sample()
    Xs = MinMaxScaler().fit_transform(Xn)
    W = kneighbors_graph(Xs, n_neighbors=max(1, min(5, len(Xs)-1)), mode='connectivity'); W = 0.5*(W+W.T)
    D = np.asarray(W.sum(axis=1)).ravel(); Dsum = D.sum()
    scores = []
    for j in range(Xs.shape[1]):
        f = Xs[:, j]; f = f - (f@D)/Dsum
        L_num = f @ (D*f) - f @ (W @ f)        # f^T L f
        L_den = f @ (D*f) + 1e-12              # f^T D f
        scores.append(L_num/L_den)
    ranked = [ALL_FEATURES[i] for i in np.argsort(scores)]   # lower score = better
    return select_k_by_cv(ranked)


In [17]:
# RFE: recursively remove the weakest feature; subset size auto-chosen by RFECV internally.
def m_rfe():
    sel = RFECV(estimator=LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
                step=1, cv=CV, scoring=SCORING).fit(X, y)

    keep = [f for f,s in zip(ALL_FEATURES, sel.support_) if s]
    return keep, f"n auto-selected by RFE with {CV_FOLDS}-fold CV (linear estimator)"


In [18]:
# RFECV: same idea with a tree estimator (captures non-linear importance), CV-chosen size.
def m_rfecv():
    sel = RFECV(estimator=RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE,
                n_jobs=N_JOBS), step=1, cv=CV, scoring=SCORING).fit(X, y)
    keep = [f for f,s in zip(ALL_FEATURES, sel.support_) if s]
    return keep, f"n auto-selected by RFECV with {CV_FOLDS}-fold CV (random forest estimator)"


In [19]:
# Sequential Forward Selection: greedily ADD features; stop at the CV-best subset.
def m_sfs():
    from mlxtend.feature_selection import SequentialFeatureSelector as MLXSFS
    s = MLXSFS(LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
               k_features='best', forward=True, floating=False, scoring=SCORING,
               cv=CV_FOLDS, n_jobs=N_JOBS).fit(X.values, y.values)
    keep = [ALL_FEATURES[i] for i in s.k_feature_idx_]
    return keep, f"CV-best forward subset (mlxtend, {CV_FOLDS}-fold {SCORING})"


In [20]:
# Sequential Backward Selection: greedily REMOVE features; stop at the CV-best subset.
def m_sbs():
    from mlxtend.feature_selection import SequentialFeatureSelector as MLXSFS
    s = MLXSFS(LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
               k_features='best', forward=False, floating=False, scoring=SCORING,
               cv=CV_FOLDS, n_jobs=N_JOBS).fit(X.values, y.values)
    keep = [ALL_FEATURES[i] for i in s.k_feature_idx_]
    return keep, f"CV-best backward subset (mlxtend, {CV_FOLDS}-fold {SCORING})"


In [21]:
# Metaheuristic wrappers (zoofs): GA / Binary PSO / Binary Grey Wolf.
# Fitness = validation MCC, matching the canonical benchmark selection objective.
def _meta(opt_cls):
    Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.25, stratify=y,
                                          random_state=RANDOM_STATE)
    from sklearn.metrics import matthews_corrcoef
    def obj(model, Xtr, ytr, Xva, yva):
        model.fit(Xtr, ytr)
        return -matthews_corrcoef(yva, model.predict(Xva))  # minimize -MCC
    opt = opt_cls(obj, n_iteration=META_ITERS, population_size=META_POP, minimize=True)
    opt.fit(LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
            Xtr, ytr, Xva, yva, verbose=False)
    return list(opt.best_feature_list)

def m_ga():
    from zoofs import GeneticOptimization
    original_pop = META_POP
    try:
        globals()['META_POP'] = X.shape[1]
        keep = _meta(GeneticOptimization)
    finally:
        globals()['META_POP'] = original_pop
    return keep, f"GA-optimized subset (fitness=val {SCORING}, {META_ITERS} iters)"

def m_bpso():
    from zoofs import ParticleSwarmOptimization
    keep = _meta(ParticleSwarmOptimization)
    return keep, f"Binary-PSO-optimized subset (fitness=val {SCORING}, {META_ITERS} iters)"

def m_bgwo():
    from zoofs import GreyWolfOptimization
    keep = _meta(GreyWolfOptimization)
    return keep, f"Binary-GWO-optimized subset (fitness=val {SCORING}, {META_ITERS} iters)"


In [22]:
# LASSO (L1-penalized logistic regression): non-zero coefficients; C tuned by CV.
def m_lasso():
    clf = LogisticRegressionCV(Cs=10, cv=CV, penalty='l1', solver='saga',
            max_iter=3000, scoring=SCORING, random_state=RANDOM_STATE, n_jobs=N_JOBS).fit(X, y)
    mask = np.abs(clf.coef_).sum(axis=0) > 1e-8
    keep = [ALL_FEATURES[i] for i in range(len(ALL_FEATURES)) if mask[i]]
    return keep, "non-zero L1 coefficients (C tuned by CV)"


In [23]:
# Elastic Net (L1+L2 logistic regression): non-zero coefficients; C and l1_ratio via CV.
def m_elasticnet():
    clf = LogisticRegressionCV(Cs=10, cv=CV, penalty='elasticnet', solver='saga',
            l1_ratios=[0.2,0.5,0.8], max_iter=3000, scoring=SCORING,
            random_state=RANDOM_STATE, n_jobs=N_JOBS).fit(X, y)
    mask = np.abs(clf.coef_).sum(axis=0) > 1e-8
    keep = [ALL_FEATURES[i] for i in range(len(ALL_FEATURES)) if mask[i]]
    return keep, "non-zero elastic-net coefficients (C and l1_ratio tuned by CV)"


In [24]:
# Tree-importance methods via SelectFromModel: keep features above MEAN importance.
def _from_model(est, label):
    sfm = SelectFromModel(est, threshold='mean').fit(X, y)
    keep = [ALL_FEATURES[i] for i in range(len(ALL_FEATURES)) if sfm.get_support()[i]]
    return keep, f"{label} importance above the mean (SelectFromModel)"
def m_rf():
    return _from_model(RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE,
            n_jobs=N_JOBS), "random-forest")

def m_extra():
    return _from_model(ExtraTreesClassifier(n_estimators=300, random_state=RANDOM_STATE,
            n_jobs=N_JOBS), "extra-trees")

def m_dtree():
    return _from_model(DecisionTreeClassifier(random_state=RANDOM_STATE), "decision-tree")


In [25]:
# XGBoost importance (gain-based) via SelectFromModel.
def m_xgb():
    from xgboost import XGBClassifier
    est = XGBClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=N_JOBS,
            tree_method='hist', eval_metric='mlogloss')
    return _from_model(est, "xgboost (gain)")


In [26]:
# LightGBM importance via SelectFromModel.
def m_lgbm():
    from lightgbm import LGBMClassifier
    est = LGBMClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=N_JOBS, verbose=-1)
    return _from_model(est, "lightgbm")


In [27]:
# Permutation importance: model-agnostic; keep features whose (mean - std) > 0 (above noise).
def m_perm():
    est = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=N_JOBS).fit(X, y)
    r = permutation_importance(est, X, y, n_repeats=PERMUTATION_REPEATS, random_state=RANDOM_STATE,
                               scoring=SCORING, n_jobs=N_JOBS)
    keep = [ALL_FEATURES[i] for i in range(len(ALL_FEATURES))
            if (r.importances_mean[i]-r.importances_std[i]) > 0]
    return keep, "features with (mean - std) permutation importance > 0 (reliably above noise)"


In [28]:
# mRMR: rank by Max-Relevance / Min-Redundancy, then CV-pick k along that ranking.
def m_mrmr():
    from mrmr import mrmr_classif
    ranked = mrmr_classif(X=X, y=y, K=len(ALL_FEATURES), show_progress=False)
    return select_k_by_cv(ranked)


In [29]:
# Boruta: all-relevant selection via shadow (shuffled) features + statistical test.
def m_boruta():
    from boruta import BorutaPy
    rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=N_JOBS)
    b = BorutaPy(rf, n_estimators='auto', max_iter=60, random_state=RANDOM_STATE, verbose=0)
    b.fit(X.values, y.values)
    keep = [ALL_FEATURES[i] for i in range(len(ALL_FEATURES)) if b.support_[i]]
    return keep, "features confirmed by Boruta's all-relevant shadow-feature test"


In [30]:
# BorutaShap: Boruta confirmation using SHAP importances instead of impurity importance.
def m_borutashap():
    # Compatibility shims: SciPy removed binom_test, NumPy 2.0 removed np.NaN. BorutaShap uses both.
    import scipy.stats as _ss
    if not hasattr(_ss, 'binom_test'):
        _ss.binom_test = (lambda x, n=None, p=0.5, alternative='two-sided':
                          _ss.binomtest(int(x), int(n), p, alternative=alternative).pvalue)
    if not hasattr(np, 'NaN'):
        np.NaN = np.nan

    import shap
    from BorutaShap import BorutaShap

    # Patch shap.TreeExplainer.shap_values to convert 3D SHAP output arrays into lists expected by BorutaShap
    orig_shap_values = shap.TreeExplainer.shap_values
    def patched_shap_values(self, X, *args, **kwargs):
        res = orig_shap_values(self, X, *args, **kwargs)
        if isinstance(res, np.ndarray) and res.ndim == 3:
            if res.shape[-1] == 2:
                return [res[:, :, 0], res[:, :, 1]]
            elif res.shape[0] == 2:
                return [res[0], res[1]]
            else:
                return [res[:, :, i] for i in range(res.shape[-1])]
        return res

    shap.TreeExplainer.shap_values = patched_shap_values
    try:
        model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=N_JOBS)
        fs = BorutaShap(model=model, importance_measure='shap', classification=True)
        Xbs, ybs = _neighbor_sample()
        fs.fit(X=Xbs, y=ybs, n_trials=50, random_state=RANDOM_STATE, verbose=False)
        keep = list(fs.accepted)
    finally:
        shap.TreeExplainer.shap_values = orig_shap_values

    return keep, "features confirmed by BorutaShap (SHAP-based all-relevant test)"



In [31]:
# SHAP ranking: mean |SHAP| from an XGBoost model; keep features covering 95% of total |SHAP|.
def m_shap_rank():
    import shap
    from xgboost import XGBClassifier
    est = XGBClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=N_JOBS,
            tree_method='hist', eval_metric='mlogloss').fit(X, y)
    sv = shap.TreeExplainer(est).shap_values(X)
    arr = np.abs(np.array(sv))           # shapes vary: (n,f), (classes,n,f), or (n,f,classes)
    nf = X.shape[1]
    # Reduce every axis except the feature axis to get one importance per feature.
    feat_axis = [ax for ax, sz in enumerate(arr.shape) if sz == nf][-1]
    other = tuple(ax for ax in range(arr.ndim) if ax != feat_axis)
    imp = arr.mean(axis=other).ravel()
    return select_by_cumulative(np.asarray(imp), frac=0.95)


## Section 7 — Dataset runner, exports, and audit

In [32]:
METHOD_ORDER=list(RUN_METHODS.keys())
METHOD_FAMILY=FAMILY
METHOD_FUNCTIONS={
 'chi2':m_chi2,'anova':m_anova,'mutual_info':m_mi,'info_gain':m_info_gain,'gain_ratio':m_gain_ratio,'fisher':m_fisher,'reliefF':m_relieff,'variance':m_variance,'spearman':m_spearman,'fcbf':m_fcbf,'laplacian':m_laplacian,
 'rfe':m_rfe,'rfecv':m_rfecv,'sfs':m_sfs,'sbs':m_sbs,'ga':m_ga,'bpso':m_bpso,'bgwo':m_bgwo,
 'lasso':m_lasso,'elasticnet':m_elasticnet,'rf_importance':m_rf,'extratrees_importance':m_extra,'dtree_importance':m_dtree,'xgb_importance':m_xgb,'lgbm_importance':m_lgbm,'permutation':m_perm,
 'mrmr':m_mrmr,'boruta':m_boruta,'borutashap':m_borutashap,'shap_rank':m_shap_rank}
assert set(METHOD_FUNCTIONS)==set(METHOD_ORDER)

def export_dataset_results():
    summary=pd.DataFrame(RESULTS)
    if len(summary):
        summary2=summary.copy(); summary2['selected_features']=summary2['selected_features'].map(lambda z:'; '.join(z))
    else: summary2=pd.DataFrame(columns=['method','family','n_selected','fit_time_seconds','justification','status'])
    summary2.to_csv(os.path.join(PATHS['results'],'nb2_fs_summary.csv'),index=False)
    mat=pd.DataFrame(0,index=METHOD_ORDER,columns=ALL_FEATURES,dtype=int)
    for r in RESULTS:
        if r['method'] in mat.index: mat.loc[r['method'],r['selected_features']]=1
    mat.to_csv(os.path.join(PATHS['results'],'nb2_selection_matrix.csv'))
    freq=mat.sum(axis=0).sort_values(ascending=False)
    freq.to_frame('times_selected').to_csv(os.path.join(PATHS['results'],'nb2_feature_frequency.csv'))
    pd.DataFrame(FAILURES).to_csv(os.path.join(PATHS['results'],'nb2_failures.csv'),index=False)
    pd.DataFrame(WARNING_LOG).to_csv(os.path.join(PATHS['results'],'nb2_warnings.csv'),index=False)
    return summary,mat,freq

def dataset_audit():
    expected=set(METHOD_ORDER); observed={r['method'] for r in RESULTS}; missing=sorted(expected-observed)
    duplicates=[m for m,n in pd.Series([r['method'] for r in RESULTS]).value_counts().items() if n>1]
    invalid=[]
    for r in RESULTS:
        invalid.extend([f for f in r['selected_features'] if f not in ALL_FEATURES])
    audit={
      'protocol_version':PROTOCOL_VERSION,'dataset':CURRENT_DATASET,'rows':len(X),'features':len(ALL_FEATURES),'expected_methods':len(expected),
      'successful_methods':len(observed),'failures':len(FAILURES),'missing_methods':missing,'duplicates':duplicates,
      'invalid_features':sorted(set(invalid)),'primary_metric':PRIMARY_METRIC,'cv_folds':CV_FOLDS,'random_state':RANDOM_STATE,
      'train_path':TRAIN_PATH,'test_exists':os.path.exists(TEST_PATH),'status':'PASS' if not missing and not duplicates and not invalid else 'INCOMPLETE'
    }
    with open(os.path.join(PATHS['results'],'nb2_master_audit.json'),'w',encoding='utf-8') as f: json.dump(audit,f,indent=2,default=str)
    print('AUDIT:',json.dumps(audit,indent=2,default=str))
    if STRICT_FINAL_AUDIT and audit['status']!='PASS': raise AssertionError(f'NB2 incomplete for {CURRENT_DATASET}: {missing}')
    return audit

def run_dataset(dataset):
    load_dataset(dataset); leakage_audit(); _load_checkpoint()
    for method in METHOD_ORDER:
        safe_run(method,METHOD_FAMILY[method],METHOD_FUNCTIONS[method])
    summary,mat,freq=export_dataset_results()
    audit=dataset_audit()
    # Lightweight figures; no fixed colors, preserving Matplotlib defaults.
    if len(summary):
        s=summary.sort_values('n_selected')
        plt.figure(figsize=(9,max(6,0.3*len(s))))
        plt.barh(s['method'],s['n_selected'])
        plt.xlabel('Number of features selected'); plt.title(f'{CURRENT_DATASET}: subset size by FS method'); plt.tight_layout()
        plt.savefig(os.path.join(PATHS['figures'],'nb2_subset_sizes.png'),dpi=150,bbox_inches='tight'); plt.close()
        top=freq.head(20)[::-1]
        plt.figure(figsize=(9,7)); plt.barh(top.index,top.values); plt.xlabel('Times selected'); plt.title(f'{CURRENT_DATASET}: feature selection frequency'); plt.tight_layout()
        plt.savefig(os.path.join(PATHS['figures'],'nb2_feature_frequency.png'),dpi=150,bbox_inches='tight'); plt.close()
    return audit


In [33]:
# ============================================================
# RUN ALL FOUR DATASETS WITH ONE IDENTICAL BASELINE PROTOCOL
# ============================================================
RUN_AUDITS=[]
for _ds in DATASETS:
    try:
        RUN_AUDITS.append(run_dataset(_ds))
    except Exception as e:
        err={'dataset':_ds,'status':'DATASET_FATAL','error_type':type(e).__name__,'error':str(e),'traceback':traceback.format_exc()}
        RUN_AUDITS.append(err)
        print(f'\nDATASET {_ds} failed at dataset level: {type(e).__name__}: {e}')
        os.makedirs(os.path.join(BENCH_ROOT,_ds,'logs'),exist_ok=True)
        with open(os.path.join(BENCH_ROOT,_ds,'logs','nb2_dataset_fatal_error.txt'),'w',encoding='utf-8') as f: f.write(err['traceback'])

manifest=pd.DataFrame(RUN_AUDITS)
manifest_path=os.path.join(BENCH_ROOT,'nb2_four_dataset_manifest.csv')
manifest.to_csv(manifest_path,index=False)
print('\nFOUR-DATASET MASTER MANIFEST')
print(manifest[['dataset','status','rows','features','successful_methods','failures']].to_string(index=False) if set(['dataset','status']).issubset(manifest.columns) else manifest.to_string(index=False))
print('Saved:',manifest_path)



[HIPAA] rows=2,634 features=32 classes={0: 558, 1: 2076}
Train: C:\Users\tanvi\Downloads\HealthSec_FS\HIPAA\processed\train.parquet | Test exists: True
Leakage/proxy flags: 0
Resumed checkpoint: 30 completed, 0 failures.
[filter  ] chi2                   -> resumed; already complete
[filter  ] anova                  -> resumed; already complete
[filter  ] mutual_info            -> resumed; already complete
[filter  ] info_gain              -> resumed; already complete
[filter  ] gain_ratio             -> resumed; already complete
[filter  ] fisher                 -> resumed; already complete
[filter  ] reliefF                -> resumed; already complete
[filter  ] variance               -> resumed; already complete
[filter  ] spearman               -> resumed; already complete
[filter  ] fcbf                   -> resumed; already complete
[filter  ] laplacian              -> resumed; already complete
[wrapper ] rfe                    -> resumed; already complete
[wrapper ] rfecv      

## Section 8 — Final scientific integrity report

In [34]:
print('='*78)
print('NB2 MASTER SCIENTIFIC INTEGRITY REPORT')
print('='*78)
print('Canonical methods :',EXPECTED_METHOD_COUNT)
print('Datasets           :',DATASETS)
print('Primary metric     :',PRIMARY_METRIC)
print('CV folds           :',CV_FOLDS)
print('Random state       :',RANDOM_STATE)
print('Resource N_JOBS    :',N_JOBS)
print('')
for a in RUN_AUDITS:
    print(a.get('dataset'), '->', a.get('status'), '| failures=',a.get('failures','NA'), '| successful=',a.get('successful_methods','NA'))

assert EXPECTED_METHOD_COUNT==30
assert len(DATASETS)==4
print('\nMASTER CONFIGURATION GATE: PASS')
print('A failure is never silently treated as a successful feature-selection result.')


NB2 MASTER SCIENTIFIC INTEGRITY REPORT
Canonical methods : 30
Datasets           : ['HIPAA', 'WUSTL', 'CICIOMT', 'WUSTL_HDRL']
Primary metric     : mcc
CV folds           : 5
Random state       : 42
Resource N_JOBS    : 1

HIPAA -> PASS | failures= 0 | successful= 30
WUSTL -> PASS | failures= 0 | successful= 30
CICIOMT -> PASS | failures= 0 | successful= 30
WUSTL_HDRL -> PASS | failures= 0 | successful= 30

MASTER CONFIGURATION GATE: PASS
A failure is never silently treated as a successful feature-selection result.
